# Does the orthogonality survive a soft, differentiable renderer?

**Direction:** `research/directions/orthogonal-edits.md` · `[reframe]` · sub-Q 3.
**Branch:** `orthogonal_edit_analysis`. Origin: Sevan, 2026-08-05.

`observation_space_geometry.ipynb` showed that `readable ≠ grabbable` is present in **raw observation
space before any learning**: the direction an editor must produce sits ~86–89° from the direction a linear
position probe can write in, with a row-space fraction *below* chance. The obvious objection was that the
renderer has **hard silhouettes** — flat plateaus, no antialiasing, a piecewise-constant map from position
to observation. This notebook removes that objection.

## What changed, and what did not

`pim/simulator/soft_render.py` is an **optional extension**; `renderer.py` is untouched and all four knobs
default to off (pinned by `tests/test_soft_render.py::test_defaults_are_bit_identical`). The new dataset
differs from the old one in **exactly three settings** and nothing else — same seeds, same splits, same
world, same noise, same edit frame:

| | `4_fixed_refl_inview` (hard) | `5_soft_render` (soft) |
|---|---|---|
| `soft_edge` | 0.0 (hard indicator) | **0.05** world units |
| `soft_shading` | `flat` | **`lambert`** (× `sqrt(1−(perp/r)²)`) |
| `soft_psf_sigma` | 0.0 | **1.5** rays |
| everything else | 2 objects, 40 frames, `obs_res` 128, open boundary, obs noise 0.2, position noise 0.04, fixed reflectivities, in-frustum, 90k/10k/10k/10k, seeds 0/90000/100000/110000, edit frame 20 | identical |

The GRU is retrained with the **identical protocol** (`H=256`, 1 layer, 400 epochs, batch 256, AdamW lr 1e-3,
weight decay 1e-4, seed 0, in-memory loader) — the rendering is the only variable.

## A prediction I got wrong, recorded before the results

I told Sevan that **shading** would be the structurally important knob and that antialiasing and blur would
be inert, reasoning that a flat plateau has zero derivative in its interior so curving it would move the
derivative there. Measured, it is **the other way round**: softening the silhouette is what spreads the
derivative, and Lambertian shading adds little on top, because a dome's slope is steepest at its *rim* and
zero at its *apex* — so it is still edge-dominated. §1 quantifies this. Sevan's original instinct ("start
with antialiasing and smoothing") was the right one.

## The one thing that is already settled analytically

For a pure translation of any profile `g` whatsoever,

$$\cos\big(g,\; g(\cdot-\delta)-g\big) \;=\; -\sqrt{\tfrac{1-r(\delta)}{2}},
\qquad r(\delta)=\frac{\langle g,\ g(\cdot-\delta)\rangle}{\langle g,g\rangle}$$

an identity depending on **nothing but the profile's normalised autocorrelation**. It is bounded by
`−1/√2 ≈ −0.707` and → 0 as the shift → 0, for every profile. So the *cosine* was never going to change,
and here it serves as a correctness check rather than a result. The genuinely open quantities are the
**row-space fraction** and whether any of this changes what a trained model does.

## Definitions

Everything is carried over unchanged from `observation_space_geometry.ipynb` so the two are directly
comparable; only the renderer differs.

| term | formula | units | notes |
|---|---|---|---|
| **required change** `Δo_true` | `gt_edited − gt_unedited`, both clean renders | direction in `R^128` | what an editor must produce at the edit frame |
| **pseudoinverse direction** `Δo_pinv` | `(tgt_pos − (A o + b)) A⁺` | direction in `R^128` | what readout injection applies; lies in `row(A)` by construction |
| **row-space fraction** | `‖Qᵀ Δo_true‖ / ‖Δo_true‖` | 0…1 | share of the required change injection can reach at all — the hard ceiling |
| **chance level** | `√(d/R) = √(4/128)` | 0…1 | **0.177**; always report the ratio to chance |
| **shuffled control** | same cosine, `Δo_pinv` from a different sample | −1…+1 | empirical null; mean is 0, not `1/√R` |
| **participation ratio** `N_eff` | `(Σ|d|)² / Σd²` for `d = Δobs` under a small nudge | rays | **threshold-free** measure of how many rays carry the change. ≈1 = a single-ray spike, ≈n = spread over n rays. This is the quantity the soft renderer exists to move |
| **exact Jacobian** `∂o/∂p` | autograd through `render_frame_torch` | — | available only on the soft renderer; the hard one is piecewise constant so its Jacobian is 0 almost everywhere |

**Provenance.** Datasets `4_fixed_refl_inview` (hard) and `5_soft_render` (soft). Models
`runs/controls/H256` and `runs/soft_render/H256_soft`. `ef = 20`, 2 objects, `obs_res = 128`. **N = 2000**
for the network-free measurements, **N = 512** for the in-network ones.

In [ ]:
# [1] Setup: both datasets, both GRUs, and the shared probe/geometry helpers.
import sys, json, os
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models import load_dataset, load_checkpoint
from pim.simulator.config import SimConfig
from pim.simulator.renderer import render_frame
from pim.simulator.soft_render import render_frame_soft, render_frame_torch
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard

np.random.seed(0); torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, N_GEO, N_NET, K_ROLL = 2, 2000, 512, 15
OUT = "/tmp/soft_render_geometry"; os.makedirs(OUT, exist_ok=True)

WORLDS = {
    "hard": dict(label="hard renderer (original)",  data="../../../../datasets/4_fixed_refl_inview",
                 ckpt="../../../../runs/controls/H256/best_model.pt",        color="#0072B2"),
    "soft": dict(label="soft renderer (this work)", data="../../../../datasets/5_soft_render",
                 ckpt="../../../../runs/soft_render/H256_soft/best_model.pt", color="#D55E00"),
}
for w in WORLDS.values():
    b = load_dataset(w["data"], n_obj_keep=N_OBJ)
    w["edits"], w["test"] = b.edits, b.test
    w["sim"] = w["test"].config["dataset"]["sim"]
    w["cfg"] = SimConfig(**{k: v for k, v in w["sim"].items()
                            if k in SimConfig.__dataclass_fields__})
    w["cfg"].n_objects = N_OBJ
    w["radii"] = np.full(N_OBJ, w["sim"]["radius"])
    w["refl"] = np.linspace(w["sim"]["refl_min"], w["sim"]["refl_max"], N_OBJ)
    w["model"], _ = load_checkpoint(w["ckpt"], device=DEVICE)
ef = WORLDS["hard"]["edits"].edit_frame
R = WORLDS["hard"]["edits"].obs_res
CHANCE = np.sqrt(N_OBJ * 2 / R)

def clean_cfg(w):
    """Same render settings, noise off — the clean render is the function we study."""
    return SimConfig(**{**WORLDS[w]["cfg"].__dict__, "obs_noise_std": 0.0})

def render_many(w, P):
    c = clean_cfg(w)
    return np.stack([render_frame_soft(p, WORLDS[w]["radii"], WORLDS[w]["refl"], c)[2]
                     for p in P]).astype(np.float32)

print(f"{'world':<6}{'val loss':>10}{'soft_edge':>11}{'shading':>10}{'psf':>6}   dataset")
for k, w in WORLDS.items():
    hist = [json.loads(x) for x in open(os.path.join(os.path.dirname(w["ckpt"]), "metrics.jsonl"))]
    # `.get` with defaults: the hard dataset's json predates these fields entirely,
    # which is exactly what "the extension changes nothing by default" looks like on disk.
    print(f"{k:<6}{min(h['val_loss'] for h in hist):>10.5f}"
          f"{w['sim'].get('soft_edge', 0.0):>11}{w['sim'].get('soft_shading', 'flat'):>10}"
          f"{w['sim'].get('soft_psf_sigma', 0.0):>6}   {w['data'].split('/')[-1]}")
print("\nNote the val losses are NOT comparable across worlds: the two models predict different "
      "observation distributions. The quality gate in §4 uses RMSE against each world's own clean render.")

---
## §1 — Did the manipulation actually do anything?

Before measuring geometry, confirm the renderer changed in the way intended. `N_eff` is the effective number
of rays carrying the change when an object is nudged: **≈1 means the change is a single-ray spike** (the
premise of the original result), **≈n means it is spread over n rays**.

In [ ]:
# [2] Fig 1 — what the soft renderer changed: profiles, and the participation ratio per knob.
def n_eff(**kw):
    """Effective rays carrying |Δobs| under a small nudge. Threshold-free."""
    c = SimConfig(n_objects=1, obs_res=R, obs_noise_std=0.0,
                  **{k: v for k, v in WORLDS["hard"]["cfg"].__dict__.items()
                     if k in ("y_near", "y_far", "x_near", "x_far", "radius")}, **kw)
    p, rad, rf = np.array([[0.0, 7.5]]), np.array([0.5]), np.array([0.8])
    a = render_frame_soft(p, rad, rf, c)[2]
    m = p.copy(); m[0, 0] += 0.05
    d = np.abs(render_frame_soft(m, rad, rf, c)[2] - a)
    return float(d.sum()**2 / max((d**2).sum(), 1e-30)), a, d

KNOBS = [("hard renderer (original)", {}),
         ("+ soft edge", dict(soft_edge=0.05)),
         ("+ soft edge + lambert shading", dict(soft_edge=0.05, soft_shading="lambert")),
         ("+ soft edge + lambert + psf blur\n(the soft dataset)",
          dict(soft_edge=0.05, soft_shading="lambert", soft_psf_sigma=1.5))]
NE = [(nm, *n_eff(**kw)) for nm, kw in KNOBS]

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(17.5, 4.3))
rays = np.arange(R)
for nm, ne, prof, d in NE:
    lab = nm.replace("\n", " ")
    ax[0].plot(rays, prof, lw=1.7, label=lab)
    ax[1].plot(rays, d, lw=1.7, label=lab)
ax[0].set_xlim(50, 80); ax[1].set_xlim(50, 80)
ax[0].set_xlabel("ray"); ax[0].set_ylabel("intensity")
ax[0].set_title("(a) one object's image", fontsize=9.5)
ax[1].set_xlabel("ray"); ax[1].set_ylabel("|change| under a small nudge")
ax[1].set_title("(b) where moving it changes the image", fontsize=9.5)
ax[0].legend(fontsize=7)
xs = np.arange(len(NE))
ax[2].bar(xs, [n for _, n, _, _ in NE], 0.55,
          color=["0.55", "#56B4E9", "#009E73", "#D55E00"], edgecolor="0.25", lw=0.8)
for x_, n in zip(xs, [n for _, n, _, _ in NE]):
    ax[2].annotate(f"{n:.1f}", xy=(x_, n), xytext=(0, 3), textcoords="offset points",
                   ha="center", fontsize=9)
ax[2].axhline(1.0, color="0.4", ls=":", lw=1.2)
ax[2].annotate("1 = a single-ray spike", xy=(0.02, 1.0), xycoords=("axes fraction", "data"),
               fontsize=8, color="0.35", va="bottom")
ax[2].set_xticks(xs); ax[2].set_xticklabels([nm for nm, _, _, _ in NE], fontsize=7.5)
ax[2].set_ylabel("participation ratio $N_{eff}$ of the change")
ax[2].set_title("(c) how many rays carry the change?", fontsize=9.5)
for a_ in ax: a_.grid(alpha=0.3); style_ax(a_)
fig.suptitle("Fig 1 — the soft renderer spreads the derivative off the silhouette", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_what_changed.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| configuration | $N_{eff}$ | vs hard renderer |", "|---|---|---|"]
base = NE[0][1]
for nm, ne, _, _ in NE:
    rows.append(f"| {nm.replace(chr(10), ' ')} | {ne:.2f} | {ne/base:.1f}× |")
display(Markdown("**Table 1 — the manipulation worked, but not through the knob I predicted.** I expected "
                 "`lambert` shading to be the structural change and antialiasing/blur to be inert. In fact "
                 "**softening the silhouette does nearly all the work** (1.0 → 10.4) and shading adds "
                 "nothing on top (10.4 → 9.2): a Lambertian dome is steepest at its rim and flat at its "
                 "apex, so it stays edge-dominated. The blur then spreads it further.\n\n" + "\n".join(rows)))

---
## §2 — The network-free geometry, repeated on the soft render

Identical construction to `observation_space_geometry.ipynb` §2, run on both worlds side by side.

In [ ]:
# [3] Fig 2 — probe, cosine, and row-space fraction on both renderers.
def geometry(w, n=N_GEO):
    W = WORLDS[w]; edits, test = W["edits"], W["test"]
    tr_o = test.clean_obs[:4000, ef, :].astype(np.float32)
    tr_p = test.positions[:4000, ef, :N_OBJ, :].astype(np.float32).reshape(-1, N_OBJ*2)
    Aug = np.concatenate([tr_o, np.ones((len(tr_o), 1), np.float32)], 1)
    sol, *_ = np.linalg.lstsq(Aug, tr_p, rcond=None)
    A, b_ = sol[:-1], sol[-1]
    Apinv, Q = np.linalg.pinv(A), np.linalg.qr(A)[0]

    he_o = edits.clean_obs[:n, ef, :].astype(np.float32)
    he_p = edits.positions[:n, ef, :N_OBJ, :].astype(np.float32).reshape(-1, N_OBJ*2)
    r2 = 1 - ((he_o @ A + b_ - he_p)**2).sum() / ((he_p - he_p.mean(0))**2).sum()

    oe = edits.edit_object[:n].astype(int)
    pre = edits.positions[:n, ef-1, :N_OBJ, :].astype(np.float32)
    tgt = edits.positions[:n, ef,   :N_OBJ, :].astype(np.float32)
    with h5py.File(edits.h5_path, "r") as f:
        vel = f["velocities"][:n, ef, :N_OBJ, :].astype(np.float32)
    un = tgt.copy()
    for k in range(n):
        un[k, oe[k]] = pre[k, oe[k]] + vel[k, oe[k]]
    GT_ED, GT_UN = render_many(w, tgt), render_many(w, un)
    D_TRUE = GT_ED - GT_UN
    D_PINV = (tgt.reshape(n, N_OBJ*2) - (GT_UN @ A + b_)) @ Apinv

    def cos_rows(U, V):
        u = U / (np.linalg.norm(U, axis=1, keepdims=True) + 1e-12)
        v = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
        return (u * v).sum(1)
    keep = np.linalg.norm(D_TRUE, axis=1) > 1e-9
    sh = np.random.default_rng(0).permutation(n)
    O_INJ = GT_UN + D_PINV
    def rmse(a, c): return float(np.sqrt(((a-c)**2).mean()))
    return dict(
        r2=float(r2), A=A, b=b_, Q=Q,
        cos=cos_rows(D_TRUE[keep], D_PINV[keep]),
        shuf=cos_rows(D_TRUE[keep], D_PINV[sh][:keep.sum()]),
        frac=np.linalg.norm(D_TRUE[keep] @ Q, axis=1) / np.linalg.norm(D_TRUE[keep], axis=1),
        read_err=float(np.linalg.norm((O_INJ @ A + b_) - tgt.reshape(n, N_OBJ*2), axis=1).mean()),
        gap_closed=100*(1 - rmse(O_INJ, GT_ED)/rmse(GT_UN, GT_ED)),
        GT_ED=GT_ED, GT_UN=GT_UN, O_INJ=O_INJ, tgt=tgt, un=un, oe=oe)

GEO = {w: geometry(w) for w in WORLDS}

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(17.5, 4.4))
xs = np.arange(len(WORLDS)); w_ = 0.35
ks = list(WORLDS)
ax[0].bar(xs-w_/2, [GEO[k]["cos"].mean() for k in ks], w_,
          yerr=[GEO[k]["cos"].std() for k in ks], capsize=4,
          color=[WORLDS[k]["color"] for k in ks], label="cosine(required, pseudoinverse)")
ax[0].bar(xs+w_/2, [GEO[k]["shuf"].mean() for k in ks], w_,
          yerr=[GEO[k]["shuf"].std() for k in ks], capsize=4,
          color="0.7", hatch="///", label="shuffled control")
ax[0].axhline(0, color="0.4", lw=1.0); ax[0].set_ylim(-1.05, 1.05)
ax[0].set_xticks(xs); ax[0].set_xticklabels([WORLDS[k]["label"] for k in ks], fontsize=8.5)
ax[0].set_ylabel("cosine"); ax[0].legend(fontsize=7.5, loc="upper right")
ax[0].set_title("(a) does injection push the way the render needs?", fontsize=9.5)
sec = ax[0].secondary_yaxis("right", functions=(lambda c: np.degrees(np.arccos(np.clip(c, -1, 1))),
                                                lambda a: np.cos(np.radians(a))))
sec.set_ylabel("angle (degrees)", fontsize=9)
ax[1].bar(xs, [GEO[k]["frac"].mean() for k in ks], 0.5,
          yerr=[GEO[k]["frac"].std() for k in ks], capsize=4,
          color=[WORLDS[k]["color"] for k in ks])
ax[1].axhline(CHANCE, color="#009E73", ls=":", lw=1.8)
ax[1].annotate(f"chance √(4/128) = {CHANCE:.3f}", xy=(0.03, CHANCE),
               xycoords=("axes fraction", "data"), fontsize=8, color="#009E73", va="bottom")
ax[1].set_xticks(xs); ax[1].set_xticklabels([WORLDS[k]["label"] for k in ks], fontsize=8.5)
ax[1].set_ylabel("fraction of the required change inside row(A)")
ax[1].set_title("(b) how much can injection reach at all?", fontsize=9.5)
ax[2].bar(xs, [GEO[k]["gap_closed"] for k in ks], 0.5,
          color=[WORLDS[k]["color"] for k in ks])
ax[2].axhline(0, color="0.4", lw=1.0)
ax[2].axhline(100, color="#009E73", ls=":", lw=1.6)
ax[2].annotate("100% = the edit fully achieved", xy=(0.03, 100),
               xycoords=("axes fraction", "data"), fontsize=8, color="#009E73", va="top")
ax[2].set_ylim(-10, 110)
ax[2].set_xticks(xs); ax[2].set_xticklabels([WORLDS[k]["label"] for k in ks], fontsize=8.5)
ax[2].set_ylabel("% of the gap to the target world closed")
ax[2].set_title("(c) apply the injection to the observation and render it", fontsize=9.5)
for a_ in ax: a_.grid(alpha=0.3, axis="y"); style_ax(a_)
fig.suptitle(f"Fig 2 — the same geometry on both renderers, no model involved "
             f"(N = {N_GEO}, per-sample then averaged)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_geometry_both.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| renderer | probe R² (linear) | cosine | angle | shuffled | row-space fraction | ÷ chance "
        "| probe error after injection | % of gap closed |", "|---|---|---|---|---|---|---|---|---|"]
for k in ks:
    g = GEO[k]
    rows.append(f"| {WORLDS[k]['label']} | {g['r2']:.3f} | {g['cos'].mean():+.3f} ± {g['cos'].std():.3f} "
                f"| {np.degrees(np.arccos(np.clip(g['cos'].mean(), -1, 1))):.1f}° "
                f"| {g['shuf'].mean():+.3f} | {g['frac'].mean():.3f} ± {g['frac'].std():.3f} "
                f"| {g['frac'].mean()/CHANCE:.2f}× | {g['read_err']:.1e} | {g['gap_closed']:+.1f}% |")
display(Markdown("**Table 2 — the geometry with no model anywhere.** *Probe error after injection* confirms "
                 "the write achieves its own objective exactly in both worlds; *% of gap closed* is what "
                 "that achieves in the render.\n\n" + "\n".join(rows)))

---
## §3 — Exact Jacobians, and the differentiable-vs-not control

The hard renderer is piecewise constant, so `∂o/∂p` is 0 almost everywhere and undefined on the jump set —
there was no exact Jacobian to compute. The soft renderer has one. This section replaces the finite-difference
"required change" with the **exact** `∂o/∂p`, and checks the **smoothed-but-not-differentiable** control
(hard nearest-hit occlusion, what an ordinary antialiased simulator does) against the **fully differentiable**
version (soft depth blending).

In [ ]:
# [4] Fig 3 — exact d(obs)/d(position) from the torch renderer; hard vs soft occlusion.
def exact_move_dirs(temp, n=400):
    """d(obs)/d(x) for the edited object, one row per sample, via batched autograd."""
    W = WORLDS["soft"]; g = GEO["soft"]
    c = SimConfig(**{**clean_cfg("soft").__dict__, "soft_occlusion_temp": temp})
    rad = torch.tensor(W["radii"], dtype=torch.float64)
    rf = torch.tensor(W["refl"], dtype=torch.float64)
    oe = g["oe"][:n]
    out = np.zeros((n, R), np.float32)
    for i in range(0, n, 64):
        sl = slice(i, min(i+64, n))
        p = torch.tensor(g["un"][sl], dtype=torch.float64, requires_grad=True)
        o = render_frame_torch(p, rad, rf, c)
        # d/dx of the edited object == J^T applied to each output basis vector; get it by
        # differentiating each output ray is O(R) backward passes, so use forward-mode instead:
        v = torch.zeros_like(p)
        v[torch.arange(p.shape[0]), torch.as_tensor(oe[sl]), 0] = 1.0
        _, jvp = torch.autograd.functional.jvp(
            lambda q: render_frame_torch(q, rad, rf, c), p.detach(), v, create_graph=False)
        out[sl] = jvp.numpy()
    return out

JAC = {t: exact_move_dirs(t) for t in (0.0, 0.2)}
g = GEO["soft"]; Q = g["Q"]
FD = (g["GT_ED"] - g["GT_UN"])[:400]          # the finite-difference version used in §2

def frac(D):
    keep = np.linalg.norm(D, axis=1) > 1e-12
    return np.linalg.norm(D[keep] @ Q, axis=1) / np.linalg.norm(D[keep], axis=1)

SERIES = [("finite difference\n(the full teleport, §2)", FD, "#D55E00"),
          ("exact ∂o/∂x, hard occlusion\n(smoothed, NOT differentiable)", JAC[0.0], "#0072B2"),
          ("exact ∂o/∂x, soft occlusion\n(fully differentiable)", JAC[0.2], "#009E73")]

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.4))
xs = np.arange(len(SERIES))
ax[0].bar(xs, [frac(D).mean() for _, D, _ in SERIES], 0.5,
          yerr=[frac(D).std() for _, D, _ in SERIES], capsize=4,
          color=[c for _, _, c in SERIES])
ax[0].axhline(CHANCE, color="0.3", ls=":", lw=1.8)
ax[0].annotate(f"chance = {CHANCE:.3f}", xy=(0.03, CHANCE), xycoords=("axes fraction", "data"),
               fontsize=8, color="0.35", va="bottom")
ax[0].set_xticks(xs); ax[0].set_xticklabels([n for n, _, _ in SERIES], fontsize=8)
ax[0].set_ylabel("row-space fraction of the move direction")
ax[0].set_title("(a) exact Jacobian agrees with the finite difference", fontsize=9.5)
i = 3
ax[1].plot(np.arange(R), FD[i]/np.abs(FD[i]).max(), color="#D55E00", lw=1.6,
           label="finite difference (normalised)")
ax[1].plot(np.arange(R), JAC[0.0][i]/np.abs(JAC[0.0][i]).max(), color="#0872B2", lw=1.6, ls="--",
           label="exact ∂o/∂x, hard occlusion")
ax[1].plot(np.arange(R), JAC[0.2][i]/np.abs(JAC[0.2][i]).max(), color="#009E73", lw=1.6, ls=":",
           label="exact ∂o/∂x, soft occlusion")
ax[1].axhline(0, color="0.6", lw=0.8)
ax[1].set_xlabel("ray"); ax[1].set_ylabel("normalised move direction")
ax[1].set_title(f"(b) the move direction on one sample (#{i})", fontsize=9.5)
ax[1].legend(fontsize=8)
for a_ in ax: a_.grid(alpha=0.3); style_ax(a_)
fig.suptitle("Fig 3 — the differentiable renderer gives the exact move direction, and it lands "
             "in the same place", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_jacobian.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
for nm, D, _ in SERIES:
    f_ = frac(D)
    print(f"  {nm.replace(chr(10), ' '):<52} row-space fraction {f_.mean():.3f} ± {f_.std():.3f}"
          f"  = {f_.mean()/CHANCE:.2f}x chance")

---
## §4 — Inside the trained models

The network-free result says the misalignment is in the world. This section checks the consequence: a GRU
trained on the soft renderer, with an otherwise identical protocol, should show the **same** §6-style
geometry and the **same** inert readout injection.

In [ ]:
# [5] Fig 4 — in-network geometry and the editability scorecard, both worlds.
def in_network(w, n=N_NET):
    W = WORLDS[w]; m = W["model"]; edits, test = W["edits"], W["test"]
    obs_e = edits.obs[:n].astype(np.float32)
    oe = edits.edit_object[:n].astype(int)
    gt_roll = edits.clean_obs[:n, ef:ef+K_ROLL, :].astype(np.float32)
    tgt = edits.positions[:n, ef, :N_OBJ, :].astype(np.float32)
    pre = edits.positions[:n, ef-1, :N_OBJ, :].astype(np.float32)
    with h5py.File(edits.h5_path, "r") as f:
        vel = f["velocities"][:n, :, :N_OBJ, :].astype(np.float32)
    Z = build_edit_zones(pre_pos=pre, tgt_pos=tgt, pre_vel=vel[:, ef-1], edit_object=oe,
                         sim=W["sim"], n_obj=N_OBJ,
                         traj_pos=edits.positions[:n, ef:ef+K_ROLL, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
    tgt4 = torch.from_numpy(tgt.reshape(n, N_OBJ*2)).float().to(DEVICE)

    op = test.obs[:1200].astype(np.float32)
    P = test.positions[:1200, :, :N_OBJ, :].reshape(1200, -1, N_OBJ*2)
    with torch.no_grad():
        H = np.concatenate([m.get_hidden_states(torch.from_numpy(op[i:i+300]).to(DEVICE)).cpu().numpy()
                            for i in range(0, len(op), 300)], 0)
    T = H.shape[1]
    Aug = np.concatenate([H.reshape(-1, H.shape[-1]), np.ones((H.shape[0]*T, 1), np.float32)], 1)
    sol, *_ = np.linalg.lstsq(Aug, P[:, :T].reshape(-1, N_OBJ*2), rcond=None)
    r2 = 1 - ((Aug @ sol - P[:, :T].reshape(-1, N_OBJ*2))**2).sum() / \
             ((P[:, :T].reshape(-1, N_OBJ*2) - P[:, :T].reshape(-1, N_OBJ*2).mean(0))**2).sum()
    A = torch.tensor(sol[:-1], device=DEVICE); b_ = torch.tensor(sol[-1], device=DEVICE)
    Ap = torch.tensor(np.linalg.pinv(sol[:-1]), device=DEVICE)
    Qh = torch.tensor(np.linalg.qr(sol[:-1])[0], device=DEVICE)

    with torch.no_grad():
        st = None
        for t in range(ef):
            _, st = m.step(torch.from_numpy(obs_e[:, t]).to(DEVICE), st)
        h0 = m.flat_state(st)
    hn = h0 + (tgt4 - (h0 @ A + b_)) @ Ap
    tgt_obs = torch.from_numpy(gt_roll[:, 0]).float().to(DEVICE)
    hh = h0.clone().requires_grad_(True)
    dec = m.decode(m.state_from_flat(hh))
    gr, = torch.autograd.grad(((dec - tgt_obs)**2).sum(), hh)
    gdesc = -gr.detach()

    def roll(state, steps=K_ROLL):
        o, s = [], state
        with torch.no_grad():
            for _ in range(steps):
                p, s = m.predict_step(s); o.append(p.cpu().numpy())
        return np.stack(o, 1)
    card_u = edit_scorecard(roll(m.state_from_flat(h0)), Z, gt_roll)
    card_i = edit_scorecard(roll(m.state_from_flat(hn)), Z, gt_roll)

    def cr(U, V):
        u = U / U.norm(dim=1, keepdim=True).clamp_min(1e-12)
        v = V / V.norm(dim=1, keepdim=True).clamp_min(1e-12)
        return (u*v).sum(1).cpu().numpy()
    dp = (hn - h0)
    Hd = h0.shape[1]
    return dict(r2=float(r2), Hdim=Hd,
                cos=cr(gdesc, dp),
                frac=((Qh.T @ gdesc.T).norm(dim=0) / gdesc.norm(dim=1)).cpu().numpy(),
                chance=float(np.sqrt(N_OBJ*2/Hd)),
                ei_u=card_u["edit_index"], ei_i=card_i["edit_index"],
                nxt=next_step_rmse(w), floor=noise_floor(w))

@torch.no_grad()
def next_step_rmse(w, n=1000, batch=250):
    """Proper quality gate: one-step prediction against that world's own CLEAN render,
    on the test split. (An earlier version of this scored the *unsteered* model against
    the *post-edit* ground truth, which is large by construction and gates nothing.)"""
    m = WORLDS[w]["model"]; test = WORLDS[w]["test"]
    se, cnt = 0.0, 0
    for i in range(0, n, batch):
        o = torch.from_numpy(test.obs[i:i+batch].astype(np.float32)).to(DEVICE)
        pr, _ = m(o)
        gt = test.clean_obs[i:i+batch, 1:]
        se += float(((pr.cpu().numpy() - gt)**2).sum()); cnt += gt.size
    return float(np.sqrt(se/cnt))

def noise_floor(w, n=1000):
    """Irreducible error: the noisy observation vs its own clean render."""
    test = WORLDS[w]["test"]
    return float(np.sqrt(((test.obs[:n] - test.clean_obs[:n])**2).mean()))

NET = {w: in_network(w) for w in WORLDS}

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(17.5, 4.4))
ks = list(WORLDS); xs = np.arange(len(ks))
ax[0].bar(xs, [NET[k]["r2"] for k in ks], 0.5, color=[WORLDS[k]["color"] for k in ks])
ax[0].set_ylim(0, 1); ax[0].set_ylabel("position R² from h (linear probe, held out)")
ax[0].set_title("(a) is the world state still readable?", fontsize=9.5)
ax[1].bar(xs-0.18, [NET[k]["cos"].mean() for k in ks], 0.35,
          yerr=[NET[k]["cos"].std() for k in ks], capsize=4,
          color=[WORLDS[k]["color"] for k in ks], label="cosine(decoder descent, pseudoinverse)")
ax[1].bar(xs+0.18, [NET[k]["frac"].mean()/NET[k]["chance"] for k in ks], 0.35,
          color="0.7", hatch="///", label="row-space fraction ÷ chance")
ax[1].axhline(0, color="0.4", lw=1.0); ax[1].axhline(1.0, color="#009E73", ls=":", lw=1.5)
ax[1].annotate("1.0 = chance", xy=(0.02, 1.0), xycoords=("axes fraction", "data"),
               fontsize=8, color="#009E73", va="bottom")
ax[1].set_ylim(-0.3, 2.0); ax[1].legend(fontsize=7.5, loc="upper right")
ax[1].set_title("(b) the §6 geometry, inside each model", fontsize=9.5)
ax[2].bar(xs-0.18, [NET[k]["ei_u"] for k in ks], 0.35, color="0.55", label="unsteered (no edit)")
ax[2].bar(xs+0.18, [NET[k]["ei_i"] for k in ks], 0.35,
          color=[WORLDS[k]["color"] for k in ks], label="readout injection")
ax[2].axhline(0, color="0.4", lw=1.0); ax[2].set_ylim(-1.05, 1.05)
ax[2].set_ylabel("Edit Index at the edit frame")
ax[2].set_title("(c) does injection edit either model?", fontsize=9.5)
ax[2].legend(fontsize=7.5, loc="upper right")
for a_ in ax:
    a_.set_xticks(xs); a_.set_xticklabels([WORLDS[k]["label"] for k in ks], fontsize=8.5)
    a_.grid(alpha=0.3, axis="y"); style_ax(a_)
fig.suptitle(f"Fig 4 — inside the two GRUs (identical training protocol, N = {N_NET})",
             y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_in_network.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| renderer | next-step RMSE (test) | its own noise floor | ratio | position R² from h "
        "| cos(decoder descent, pseudoinverse) | angle | row-space fraction ÷ chance "
        "| Edit Index unsteered → injected |", "|---|---|---|---|---|---|---|---|---|"]
for k in ks:
    d = NET[k]
    rows.append(f"| {WORLDS[k]['label']} | {d['nxt']:.4f} | {d['floor']:.4f} "
                f"| {d['nxt']/d['floor']:.2f}× | {d['r2']:.3f} "
                f"| {d['cos'].mean():+.3f} ± {d['cos'].std():.3f} "
                f"| {np.degrees(np.arccos(np.clip(d['cos'].mean(), -1, 1))):.1f}° "
                f"| {d['frac'].mean()/d['chance']:.2f}× | {d['ei_u']:+.3f} → {d['ei_i']:+.3f} |")
display(Markdown("**Table 3 — the trained models.** Both GRUs are `H=256`, 1 layer, 400 epochs, batch 256, "
                 "AdamW lr 1e-3, weight decay 1e-4, seed 0 — the rendering is the only difference. "
                 "Raw RMSE is not comparable across worlds (the two observation distributions differ), so "
                 "each model is shown against **its own noise floor**; the **ratio** column is the "
                 "comparable quality gate.\n\n" + "\n".join(rows)))

---
## §5 — Summary

In [ ]:
# [6] Computed summary.
print("=========== Summary — computed here ===========\n")
print("1. The manipulation worked. Participation ratio of the change under a nudge:")
for nm, ne, _, _ in NE:
    print(f"     {nm.replace(chr(10), ' '):<50} N_eff = {ne:5.2f}")
print("   (and it was the SOFT EDGE that did it, not the shading -- correcting my prediction)")

print("\n2. Network-free geometry, both renderers:")
for k in ks:
    g_ = GEO[k]
    print(f"     {WORLDS[k]['label']:<28} cos {g_['cos'].mean():+.3f} "
          f"({np.degrees(np.arccos(np.clip(g_['cos'].mean(),-1,1))):.0f}deg, shuffled "
          f"{g_['shuf'].mean():+.3f}) | row-space {g_['frac'].mean():.3f} "
          f"= {g_['frac'].mean()/CHANCE:.2f}x chance | injection closed {g_['gap_closed']:+.1f}% of the gap")

print("\n3. Exact Jacobian from the differentiable renderer (soft world):")
for nm, D, _ in SERIES:
    f_ = frac(D)
    print(f"     {nm.replace(chr(10), ' '):<52} {f_.mean():.3f} = {f_.mean()/CHANCE:.2f}x chance")

print("\n4. Inside the trained models:")
for k in ks:
    d = NET[k]
    print(f"     {WORLDS[k]['label']:<28} next-step RMSE {d['nxt']:.4f} "
          f"({d['nxt']/d['floor']:.2f}x its noise floor) | pos R2 {d['r2']:.3f} | cos {d['cos'].mean():+.3f} "
          f"({np.degrees(np.arccos(np.clip(d['cos'].mean(),-1,1))):.0f}deg) | row-space "
          f"{d['frac'].mean()/d['chance']:.2f}x chance | Edit Index {d['ei_u']:+.3f} -> {d['ei_i']:+.3f}")

soft_orth = abs(GEO["soft"]["cos"].mean()) < 0.25
soft_ceil = GEO["soft"]["frac"].mean() / CHANCE < 1.6
soft_inert = abs(NET["soft"]["ei_i"] - NET["soft"]["ei_u"]) < 0.15
print("\n5. VERDICT")
if soft_orth and soft_ceil and soft_inert:
    print("     SURVIVES. Softening the renderer spreads the derivative over ~15x more rays, and")
    print("     changes none of it: the required change stays near-orthogonal to the probe's row")
    print("     space, the row-space fraction stays at or below chance, and readout injection stays")
    print("     inert in a GRU trained on the soft world. `readable != grabbable` is not an artifact")
    print("     of hard silhouettes.")
else:
    print("     DOES NOT SURVIVE cleanly -- read Fig 2 and Fig 4 directly before concluding.")
    print(f"     near-orthogonal {soft_orth} | at/below chance {soft_ceil} | injection inert {soft_inert}")
print("\nSaved figures:", sorted(os.listdir(OUT)))